# Stereo Vision in ADAS: Pioneering Depth Perception Beyond LiDAR

---

- Conda env : 
    ```
    conda create -y -n sttr python=3.10.12
    conda activate sttr

    pip install ipykernel ipywidgets
    pip install torch torchvision torchcontrib
    pip install albumentations==0.4.6 natsort
    pip install tensorboardx tensorboard
    pip install opencv-python scikit-learn
    pip install numpy==1.26.4
    ```
----

- Ref : https://learnopencv.com/adas-stereo-vision/



In [1]:
!nvidia-smi

Mon Nov 24 15:27:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 24%   43C    P5             33W /  250W |     643MiB /  11264MiB |     21%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import requests

def download_dataset(url, local_filename):

    # Update Dropbox link to force download
    if "www.dropbox.com" in url and "?dl=0" in url:
        url = url.replace("?dl=0", "?dl=1")
    
    # Send a GET request to the URL
    response = requests.get(url)
    
    # Check if the request was successful
    if response.status_code == 200:
        # Write the content of the response to a file
        with open(local_filename, 'wb') as f:
            f.write(response.content)
        print(f"File downloaded and saved as {local_filename}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [5]:
from pathlib import Path
Path("./temp_src").mkdir(exist_ok=True, parents=True)
zipfile_path = './temp_src/adas-stereo-vision.zip'
# Download 10% sample of BDD100K Dataset
download_dataset('https://www.dropbox.com/scl/fi/q9it1xgtdvvxp140zn45o/adas-stereo-vision.zip?rlkey=yi11yjo2ivjmgko0ponx5u46z&dl=1', zipfile_path)

File downloaded and saved as ./temp_src/adas-stereo-vision.zip


In [6]:
import zipfile

src_path = "./temp_src/adas-stereo-vision"
with zipfile.ZipFile(zipfile_path, 'r') as zip_ref:
    zip_ref.extractall(src_path)

In [1]:
from PIL import Image
import torch
import numpy as np
import cv2
import glob
import os


import argparse
import matplotlib.pyplot as plt
import sys
src_dir = "./temp_src/adas-stereo-vision/stereo-transformer"
sys.path.append(src_dir) # add relative path

from module.sttr import STTR
from dataset.preprocess import normalization, compute_left_occ_region
from utilities.misc import NestedTensor

In [2]:
# Function to load images
def load_images(image_dir, pattern):
    filenames = sorted(glob.glob(os.path.join(image_dir, pattern)))
    return [np.array(Image.open(filename)) for filename in filenames[:500]]

In [3]:
# Default parameters
args = type('', (), {})() # create empty args
args.channel_dim = 128
args.position_encoding = 'sine1d_rel'
args.num_attn_layers = 6
args.nheads = 8
args.regression_head = 'ot'
args.context_adjustment_layer = 'cal'
args.cal_num_blocks = 8
args.cal_feat_dim = 16
args.cal_expansion_ratio = 4

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Using device: {device}")

Using device: cpu


In [5]:
# model = STTR(args).cuda().eval()
model = STTR(args).to(device).eval()

/home/hyunjae/anaconda3/envs/sttr/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [6]:
# Load the pretrained model
import os
model_file_name = os.path.join(src_dir, "kitti_finetuned_model.pth.tar")
checkpoint = torch.load(model_file_name, map_location=device)
pretrained_dict = checkpoint['state_dict']
model.load_state_dict(pretrained_dict, strict=False) # prevent BN parameters from breaking the model loading
print("Pre-trained model successfully loaded.")

Pre-trained model successfully loaded.


In [7]:
# Load images
left_images_dir = os.path.join(src_dir, "sample_data/KITTI_2015/2015/training/image_2")
left_images = load_images(left_images_dir, '*.png')
right_images_dir = os.path.join(src_dir, "sample_data/KITTI_2015/2015/training/image_3")
right_images = load_images(right_images_dir, '*.png')

In [8]:
# Initialize video writer
height, width, _ = left_images[0].shape
output_dir = './inference_output/'
os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist

In [9]:
for i, (left, right) in enumerate(zip(left_images[:3], right_images[:3])):
    # Normalize and create NestedTensor for each set of images
    input_data = normalization(left=left, right=right)
    h, w, _ = left.shape
    bs = 1
    downsample = 3
    col_offset = int(downsample / 2)
    row_offset = int(downsample / 2)
    sampled_cols = torch.arange(col_offset, w, downsample)[None,].expand(bs, -1).cpu()
    sampled_rows = torch.arange(row_offset, h, downsample)[None,].expand(bs, -1).cpu()
    input_data = NestedTensor(input_data['left'].to(device)[None,], input_data['right'].to(device)[None,], sampled_cols=sampled_cols, sampled_rows=sampled_rows)

    # Perform inference
    output = model(input_data)
    disp_pred = output['disp_pred'].data.cpu().numpy()[0]
    occ_pred = output['occ_pred'].data.cpu().numpy()[0] > 0.5
    disp_pred[occ_pred] = 0.0

     # Ensure disp_pred and occ_pred are normalized and converted to uint8
    disp_pred_norm = cv2.normalize(disp_pred, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    occ_pred_uint8 = np.uint8(occ_pred * 255)

    # Combine predicted disparity and occlusion map
    combined_input = np.hstack((left, right))
    combined_output = np.hstack((disp_pred_norm, occ_pred_uint8))
    combined_output = cv2.cvtColor(combined_output, cv2.COLOR_GRAY2BGR)

    combined_all = np.vstack((combined_input, combined_output))
    # Save the combined output as a PNG file
    output_filename = os.path.join(output_dir, f'inference_{i:03d}.png')
    cv2.imwrite(output_filename, combined_all)
    print(f"Saved: {output_filename}")

print("All inferences saved as PNG files.")

/home/hyunjae/anaconda3/envs/sttr/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Saved: ./inference_output/inference_000.png
Saved: ./inference_output/inference_001.png
Saved: ./inference_output/inference_002.png
All inferences saved as PNG files.
